# Laboratorio de regresión - 4

|                |   |
:----------------|---|
| **Nombre**     |  David Farias Rivera |
| **Fecha**      |  18/10/2026 |
| **Expediente** |  754381 |

## Modelos penalizados

Hasta ahora la función de costo que usamos para decidir qué tan bueno es nuestro modelo al momento de ajustar es:

$$ \text{RSS} = \sum_{i=1}^n e_i^2 = \sum_{i=1}^n (y_i - \hat{y_i})^2 $$

Dado que los errores obtenidos son una combinación de sesgo y varianza, puede ser que se sesgue un parámetro para minimizar el error. Esto significa que el modelo puede decidir que la salida no sea una combinación de los factores, sino una fuerte predilección sobre uno de los factores solamente. 

E.g. se quiere ajustar un modelo

$$ \hat{z} = \hat{\beta_0} + \hat{\beta_1} x + \hat{\beta_2} y $$

Se ajusta el modelo y se decide que la mejor decisión es $\hat{\beta_1} = 10000$ y $\hat{\beta_2}=50$. Considera limitaciones de problemas reales:
- Quizás los parámetros son ajustes de maquinaria que se deben realizar para conseguir el mejor producto posible, y que $10000$ sea imposible de asignar.
- Quizás los datos actuales están sesgados y sólo hacen parecer que uno de los factores importa más que el otro.

Una de las formas en las que se puede mitigar este problema es penalizando a los parámetros del modelo, cambiando la función de costo:

$$ \text{RSS}_{L2} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p \hat{\beta_j}^2 $$

El *L2* significa que se está agregando una penalización de segundo orden. Lo que hace esta penalización es que los factores ahora sólo tendrán permitido crecer si hay una reducción al menos proporcional en el error (sacrificamos sesgo, pero reducimos la varianza).

Asimismo, existe la penalización *L1*

$$ \text{RSS}_{L1} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p |\hat{\beta_j}| $$

A las penalizaciones *L2* y *L1* se les conoce también como Ridge y Lasso, respectivamente.

Para realizar una regresión con penalización de Ridge o de Lasso usamos el objeto `Ridge(alpha=?)` o `Lasso(alpha=?)` en lugar de `LinearRegression()` de `sklearn`.

Utiliza el dataset de publicidad (Advertising.csv), utiliza train-test-split de 70/30 y realiza 3 regresiones múltiples:

$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$

1. Sin penalización
2. Con penalización L2
3. Con penalización L1

¿Qué puedes observar al ajustar los valores de `alpha`? 

Compara los resultados de los coeficientes utilizando valores diferentes de $\alpha$ y los $R^2$ resultantes.



In [10]:
import pandas as pd  
from sklearn.linear_model import LinearRegression, Ridge, Lasso 
from sklearn.model_selection import train_test_split  
from sklearn.preprocessing import StandardScaler  

data = pd.read_csv("/Users/david/Documents/Codigod/ITESO/01-ITESO/Lab 3/Advertising.csv", index_col=0, header=0)  # Cargamos los datos, la primera columna es el numero de fila
x = data[["TV", "radio", "newspaper"]]  # Asignamos nuestra X 
y = data["sales"]  # Asignamos nuestra Y 

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0)  # 70% para entrenar y 30% para probar

scaler = StandardScaler()  # Creamos el escalador
x_train_s = scaler.fit_transform(x_train)  # Aprende media y desviacion solo con train y lo escala
x_test_s = scaler.transform(x_test)  # Escala test con lo que aprendio de train, sin volver a ajustar
x_train_s.shape, x_test_s.shape


((140, 3), (60, 3))

In [11]:
# Creamos el modelo sin penalizacion

lr = LinearRegression()
lr.fit(x_train_s, y_train)  # Lo entrenamos con train escalado

print(lr.coef_) # Mostramos los coeficientes del modelo sin penalización
print(lr.score(x_train_s, y_train)) # Mostramos el R^2 del modelo sin penalización
print(lr.score(x_test_s, y_test)) # Mostramos el R^2 del modelo sin penalización en el conjunto de prueba


[3.68471841 3.05065643 0.03809335]
0.9065727532450596
0.8649018906637791


In [ ]:
for i in [0.1, 1, 10, 100]:
    ridge = Ridge(alpha=i)  #Creamos el modelo con penalización Ridge
    ridge.fit(x_train_s, y_train)  

    print("alpha =", i) 
    print("Coeficientes:", ridge.coef_) 
    print("R2 train:", ridge.score(x_train_s, y_train))
    print("R2 test:", ridge.score(x_test_s, y_test)) 
    print()


alpha = 0.1
Coeficientes: [3.68208065 3.04832412 0.03908188]
R2 train: 0.9065723027922852
R2 test: 0.864878434938684

alpha = 1
Coeficientes: [3.65851755 3.02751315 0.04784104]
R2 train: 0.9065283461721827
R2 test: 0.8646285057422319

alpha = 10
Coeficientes: [3.4391112  2.83567945 0.12329061]
R2 train: 0.9026977963826662
R2 test: 0.8588514245483714

alpha = 100
Coeficientes: [2.16384426 1.77085802 0.35302732]
R2 train: 0.760404368497697
R2 test: 0.70673919423062



In [1]:
for i in [0.01, 0.1, 1, 3]:
    lasso = Lasso(alpha=i) #Creamos el modelo con penalización Lasso
    lasso.fit(x_train_s, y_train)  

    print("alpha =", i) 
    print("Coeficientes:", lasso.coef_) 
    print("R2 train:", lasso.score(x_train_s, y_train)) 
    print("R2 test:", lasso.score(x_test_s, y_test)) 
    print()


NameError: name 'Lasso' is not defined

- **Escalé con StandardScaler para que el castigo fuera parejo entre variables. Al subir alpha los coeficientes se hacen más chicos y el R2 baja. Ridge encoge todo pero nunca llega a cero, Lasso sí: con alpha de 0.1 deja newspaper en cero sin perder R2, igual que en el lab 2 donde no era significativo. Aquí penalizar no mejora porque el modelo no está sobreajustado.**